# 07 — silver_order_payments
**Entrée :** `bronze.order_payments` — **Sortie :** `silver.silver_order_payments` (**103 886 lignes**)
**Typage :** `payment_value` en `DECIMAL(10,2)`, numéro de paiement et échéances en entier.
**Indicateurs :** `flag_zero_value` (9 paiements à 0), `flag_not_defined` (type `not_defined`).
**Clé :** (`order_id`, `payment_sequential`).
**Rappel (R16) :** les 380 écarts entre total des articles et total payé se **marquent en Gold**, on ne les corrige pas ici.

In [0]:
CATALOG = "workspace"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")


def scalar(sql):
    return spark.sql(sql).collect()[0][0]


def check(label, obtenu, attendu):
    ok = obtenu == attendu
    print(("✅ OK     " if ok else "❌ ÉCART  ") + f"{label} : obtenu = {obtenu} | attendu = {attendu}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {SILVER}.silver_order_payments AS
WITH p AS (
    SELECT
        order_id,
        CAST(TRY_CAST(payment_sequential   AS DOUBLE) AS INT) AS payment_sequential,
        LOWER(TRIM(payment_type))                             AS payment_type,
        CAST(TRY_CAST(payment_installments AS DOUBLE) AS INT) AS payment_installments,
        TRY_CAST(payment_value AS DECIMAL(10,2))              AS payment_value,
        _loaded_at
    FROM {BRONZE}.order_payments
)
SELECT
    p.*,
    COALESCE(payment_value = 0, FALSE)              AS flag_zero_value,
    COALESCE(payment_type = 'not_defined', FALSE)   AS flag_not_defined,
    _loaded_at AS _bronze_loaded_at,
    current_timestamp() AS _silver_loaded_at
FROM p
""")
print("silver_order_payments créée")

silver_order_payments créée


In [0]:
t = f"{SILVER}.silver_order_payments"
check("lignes", scalar(f"SELECT COUNT(*) FROM {t}"), 103886)
check("commandes distinctes", scalar(f"SELECT COUNT(DISTINCT order_id) FROM {t}"), 99440)
check("clé (order_id, payment_sequential) en double", scalar(
    f"SELECT COUNT(*) - COUNT(DISTINCT order_id, payment_sequential) FROM {t}"), 0)
check("valeur NULL (conversion échouée)", scalar(f"SELECT COUNT(*) FROM {t} WHERE payment_value IS NULL"), 0)
check("flag_zero_value", scalar(f"SELECT COUNT(*) FROM {t} WHERE flag_zero_value"), 9)
check("flag_not_defined", scalar(f"SELECT COUNT(*) FROM {t} WHERE flag_not_defined"), 3)
check("types de paiement distincts", scalar(f"SELECT COUNT(DISTINCT payment_type) FROM {t}"), 5)

s_b = scalar(f"SELECT ROUND(SUM(TRY_CAST(payment_value AS DOUBLE)), 2) FROM {BRONZE}.order_payments")
s_s = float(scalar(f"SELECT SUM(payment_value) FROM {t}"))
check("somme des paiements Bronze = Silver", round(s_s, 2), round(s_b, 2))

✅ OK     lignes : obtenu = 103886 | attendu = 103886
✅ OK     commandes distinctes : obtenu = 99440 | attendu = 99440
✅ OK     clé (order_id, payment_sequential) en double : obtenu = 0 | attendu = 0
✅ OK     valeur NULL (conversion échouée) : obtenu = 0 | attendu = 0
✅ OK     flag_zero_value : obtenu = 9 | attendu = 9
✅ OK     flag_not_defined : obtenu = 3 | attendu = 3
✅ OK     types de paiement distincts : obtenu = 5 | attendu = 5
✅ OK     somme des paiements Bronze = Silver : obtenu = 16008872.12 | attendu = 16008872.12


## À regarder (R17, à documenter)
Les échéances par moyen de paiement : un paiement `boleto` ou `debit_card` avec plusieurs échéances est-il normal ?

In [0]:
spark.sql(f"""
    SELECT payment_type, COUNT(*) AS nb, MIN(payment_installments) AS min_ech,
           MAX(payment_installments) AS max_ech, ROUND(AVG(payment_installments), 2) AS moy_ech
    FROM {SILVER}.silver_order_payments GROUP BY payment_type ORDER BY nb DESC
""").show()

+------------+-----+-------+-------+-------+
|payment_type|   nb|min_ech|max_ech|moy_ech|
+------------+-----+-------+-------+-------+
| credit_card|76795|      0|     24|   3.51|
|      boleto|19784|      1|      1|    1.0|
|     voucher| 5775|      1|      1|    1.0|
|  debit_card| 1529|      1|      1|    1.0|
| not_defined|    3|      1|      1|    1.0|
+------------+-----+-------+-------+-------+



## Pour ton journal (à rédiger avec tes mots)
1. Pourquoi `TRY_CAST` plutôt que `CAST` ?
2. Pourquoi on garde les lignes anormales avec un indicateur `flag_…` ?
3. Pourquoi Silver a le même nombre de lignes que Bronze ?